# Update Barton Springs WEL with interactive selection

This notebook loads the Barton Springs WEL file and the EBFZ grid, shows the grid in Plotly,
and lets you select cells (box/lasso/click) to set a new pumping rate. Click **Apply + Save**
to write a new WEL file locally.


In [ ]:
import shutil
import zipfile
from pathlib import Path
import urllib.request

import flopy
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Polygon, MultiPolygon

# --- downloads ---
wel_url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/9c7b25c4-8cea-4965-a07a-d9b3867f18a9/"
    "download/barton_springs_2001_2010average.wel"
)
grid_url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/f07a257c-1d88-4819-bd5d-a104c5e3fe5b/"
    "download/ebfz_b_grid.zip"
)

wel_path = Path("barton_springs_2001_2010average.wel")
grid_zip = Path("ebfz_b_grid.zip")
grid_dir = Path("ebfz_b_grid")
grid_gdb = grid_dir / "ebfz_b_grid.gdb"
grid_layer = "ebfz_b_grid_poly101223"

def _flatten_single_dir(root: Path) -> None:
    children = [p for p in root.iterdir() if p.is_dir()]
    if len(children) == 1:
        inner = children[0]
        for item in inner.iterdir():
            shutil.move(str(item), root)
        inner.rmdir()

if not wel_path.exists():
    urllib.request.urlretrieve(wel_url, wel_path)

if not grid_gdb.exists():
    urllib.request.urlretrieve(grid_url, grid_zip)
    grid_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(grid_zip, "r") as zf:
        zf.extractall(grid_dir)
    _flatten_single_dir(grid_dir)

# --- load grid ---
gdf = gpd.read_file(grid_gdb, layer=grid_layer)
gdf = gdf.to_crs("EPSG:4326")
gdf["_centroid"] = gdf.geometry.centroid
gdf["_lon"] = gdf["_centroid"].x
gdf["_lat"] = gdf["_centroid"].y

# --- scan WEL for grid dimensions ---
def scan_wel_metadata(path: Path):
    def strip_comment(line):
        for token in ("#", ";"):
            if token in line:
                line = line.split(token, 1)[0]
        return line.strip()

    lines = [strip_comment(line) for line in path.read_text().splitlines()]
    data_lines = [line for line in lines if line]
    data_lines.pop(0)
    nper = 0
    max_k = max_i = max_j = 1
    idx = 0
    while idx < len(data_lines):
        tokens = data_lines[idx].split()
        idx += 1
        if not tokens:
            continue
        nper += 1
        itmp = int(tokens[0])
        if itmp <= 0:
            continue
        for _ in range(itmp):
            parts = data_lines[idx].split()
            idx += 1
            k, i, j = (int(parts[0]), int(parts[1]), int(parts[2]))
            max_k = max(max_k, k)
            max_i = max(max_i, i)
            max_j = max(max_j, j)
    return nper, max_k, max_i, max_j

nper, nlay, nrow, ncol = scan_wel_metadata(wel_path)

m = flopy.modflow.Modflow(modelname="wel_read", model_ws=".")
flopy.modflow.ModflowDis(
    m,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    nper=nper,
    delr=1.0,
    delc=1.0,
    top=1.0,
    botm=[0.0] * nlay,
)
wel = flopy.modflow.ModflowWel.load(str(wel_path), m)


In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

def _polygon_outline_coords(geom):
    lons, lats = [], []
    if isinstance(geom, Polygon):
        xs, ys = geom.exterior.coords.xy
        lons.extend(xs)
        lats.extend(ys)
        lons.append(None)
        lats.append(None)
    elif isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            xs, ys = part.exterior.coords.xy
            lons.extend(xs)
            lats.extend(ys)
            lons.append(None)
            lats.append(None)
    return lons, lats

outline_lons, outline_lats = [], []
for geom in gdf.geometry:
    xs, ys = _polygon_outline_coords(geom)
    outline_lons.extend(xs)
    outline_lats.extend(ys)

fig = go.FigureWidget()
fig.add_trace(
    go.Scattergeo(
        lon=outline_lons,
        lat=outline_lats,
        mode="lines",
        line=dict(color="#666", width=1),
        hoverinfo="skip",
        name="Grid",
    )
)
fig.add_trace(
    go.Scattergeo(
        lon=gdf["_lon"],
        lat=gdf["_lat"],
        mode="markers",
        marker=dict(size=4, color="#1f77b4"),
        name="Cells",
        customdata=np.stack([gdf["CELL_ID"], gdf["ROW"], gdf["COL"]], axis=1),
        hovertemplate="CELL_ID=%{customdata[0]}<br>ROW=%{customdata[1]} COL=%{customdata[2]}<extra></extra>",
        selected=dict(marker=dict(size=6, color="#d62728")),
        unselected=dict(marker=dict(opacity=0.5)),
    )
)
fig.update_geos(fitbounds="locations", showcountries=False, showcoastlines=False)
fig.update_layout(
    height=600,
    margin=dict(l=0, r=0, t=0, b=0),
    dragmode="lasso",
)

selected_ids = set()
status = widgets.Label(value="Selected cells: 0")

def _update_selected(points):
    selected_ids.clear()
    for p in points:
        cid = int(p["customdata"][0])
        selected_ids.add(cid)
    status.value = f"Selected cells: {len(selected_ids)}"

def _on_selection(trace, points, state):
    _update_selected(points.points)

def _on_click(trace, points, state):
    for p in points.points:
        cid = int(p.customdata[0])
        if cid in selected_ids:
            selected_ids.remove(cid)
        else:
            selected_ids.add(cid)
    status.value = f"Selected cells: {len(selected_ids)}"

fig.data[1].on_selection(_on_selection)
fig.data[1].on_click(_on_click)

rate_input = widgets.FloatText(value=-20.0, description="New rate")
layer_input = widgets.IntText(value=1, description="Layer (k)")
add_missing = widgets.Checkbox(value=False, description="Add missing wells")
save_btn = widgets.Button(description="Apply + Save", button_style="primary")
output = widgets.Output()

def _apply_and_save(_):
    with output:
        clear_output()
        if not selected_ids:
            print("No cells selected.")
            return
        spd = wel.stress_period_data.data
        cell_lookup = dict(zip(gdf["CELL_ID"], zip(gdf["ROW"], gdf["COL"])))
        selected_cells = {cell_lookup[cid] for cid in selected_ids if cid in cell_lookup}
        new_spd = {}
        for per, recs in spd.items():
            recs = recs.copy()
            mask = []
            for rec in recs:
                i = int(rec["i"])
                j = int(rec["j"])
                mask.append((i, j) in selected_cells)
            mask = np.array(mask, dtype=bool)
            recs["flux"][mask] = float(rate_input.value)
            if add_missing.value and selected_cells:
                existing = set((int(r["i"]), int(r["j"])) for r in recs)
                to_add = [cell for cell in selected_cells if cell not in existing]
                if to_add:
                    new_recs = np.zeros(len(recs) + len(to_add), dtype=recs.dtype)
                    new_recs[: len(recs)] = recs
                    for idx, (row, col) in enumerate(to_add, start=len(recs)):
                        new_recs[idx]["k"] = int(layer_input.value)
                        new_recs[idx]["i"] = int(row)
                        new_recs[idx]["j"] = int(col)
                        new_recs[idx]["flux"] = float(rate_input.value)
                    recs = new_recs
            new_spd[per] = recs
        wel.stress_period_data = new_spd
        updated_wel = Path("barton_springs_updated.wel")
        wel.write_file(str(updated_wel))
        print(f"Updated {len(selected_cells)} cells. Wrote {updated_wel}")

save_btn.on_click(_apply_and_save)

display(widgets.VBox([
    fig,
    widgets.HBox([rate_input, layer_input, add_missing, save_btn]),
    status,
    output,
]))
